# Probabilistic modelling of 32 Gbaud Nyquist-WDM 16-QAM optical communication system

## Libraries and modules
We import standard libraries for data manipulation (`pandas`, `numpy`) and visualization (`matplotlib`). Crucially, we use `scipy.stats.gaussian_kde` for the core feature extraction and `scipy.signal.find_peaks` for topological analysis.

In [1]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

from scipy.stats import gaussian_kde
from scipy.signal import find_peaks
from dotenv import load_dotenv

import warnings
import os
import gdown

## Database loading
We load the dataset from a remote Parquet file (if not locally present). This dataset contains raw simulated I/Q symbols for various optical transmission scenarios.

In [12]:
# Load environment variables for DB GDrive ID
load_dotenv()
FILE_ID = os.getenv("GDRIVE_FILE_ID")
if not FILE_ID:
    warnings.warn("GDRIVE_FILE_ID not set, not downloading.")
DATABASE_FILENAME = "./data/32gbaud.parquet"

if not os.path.exists(DATABASE_FILENAME) and FILE_ID:
    print("Downloading data from Google Drive...")
    os.makedirs(os.path.dirname(DATABASE_FILENAME), exist_ok=True)
    url = f"https://drive.google.com/uc?id={FILE_ID}"
    gdown.download(url, DATABASE_FILENAME, quiet=False)
    print("Download successful!")
else:
    print("Data already downloaded! skipping...")

Downloading...
From (original): https://drive.google.com/uc?id=1LR68DFw6CaqqUJFyyDK2SI5JMHo6bvTD
From (redirected): https://drive.google.com/uc?id=1LR68DFw6CaqqUJFyyDK2SI5JMHo6bvTD&confirm=t&uuid=6b40c72c-1e9f-4c8e-9b92-737b000d596b
To: /home/cesar/Academy/UdeA/Semestre/Práctica académica/Inferencia estadística/proyecto/mogp_with_features/data/32gbaud.parquet
100%|██████████| 802M/802M [12:14<00:00, 1.09MB/s] 

Download successful!


In [9]:
print("Loading data...")
df = pd.read_parquet(DATABASE_FILENAME)
print("Data loaded!")

Loading data...
Data loaded!


### Data Preprocessing: Batch Generation

In this section, we process the raw dataframe to create fixed-size batches of symbols for each specific scenario. Since the feature extraction method benefits from having inputs of exact dimensions, we must handle the variable number of symbols per scenario.

**Key steps:**
1.  **Grouping:** We identify unique scenarios based on `Distance`, `Power`, `Spacing`, and `OSNR`.
2.  **Filtering:** We discard any "leftover" data that does not fit into a complete batch of 10,000 symbols (e.g., if a scenario has 38k symbols, the last 8k are dropped).
3.  **Indexing:** We assign a unique `batch_number` to every 10,000 symbols within each scenario.

> **Note:** This operation is fully vectorized using Pandas `cumcount` and `transform` to handle the large dataset efficiently, avoiding slow Python loops.

In [10]:
BATCH_SIZE = 10_000
GROUP_COLS = ["Distance", "Power", "Spacing", "OSNR"]

# 1. Group by scenario characteristics
g = df.groupby(GROUP_COLS)

# 2. Generate a sequential counter for each row WITHIN its group
# This assigns 0, 1, 2... N to the rows of each specific scenario
df["row_id"] = g.cumcount()

# 3. Calculate total rows per group
# Use transform so the result has the same length as the original df
df["group_count"] = g["I"].transform("count")

# 4. Filter data
# Logic: Keep the row only if its position (row_id) is less than the limit
# of the last full batch.
# (Integer division removes decimals, multiply by batch size to get the cutoff)
mask = df["row_id"] < (df["group_count"] // BATCH_SIZE) * BATCH_SIZE
df_batches = df[mask].copy()

# 5. Assign batch number
# e.g., rows 0-9999 -> batch 0; rows 10000-19999 -> batch 1
df_batches["batch_number"] = df_batches["row_id"] // BATCH_SIZE

# Cleanup: Drop auxiliary columns to free up memory
df_batches.drop(columns=["row_id", "group_count"], inplace=True)

print(f"Original rows: {len(df)}")
print(f"Processed rows (full batches only): {len(df_batches)}")
print(df_batches.head())

Original rows: 51825141
Processed rows (full batches only): 50060000
          I         Q  Distance  Power   OSNR  Spacing  batch_number
0  1.125429  1.909534       0.0    0.0  23.04     32.0             0
1 -1.502832  1.121298       0.0    0.0  23.04     32.0             0
2  1.636159  0.695216       0.0    0.0  23.04     32.0             0
3  2.839414  3.112252       0.0    0.0  23.04     32.0             0
4  0.979563  2.873738       0.0    0.0  23.04     32.0             0


### Data Organization: Raw Symbols and Targets

In this step, we restructure the sorted DataFrame into NumPy arrays. This aligns the raw I/Q symbol sequences with their corresponding scenario labels, preparing the data for the subsequent feature extraction stage.

1.  **X (Raw Data):** A 3D array of shape `(N_batches, Sequence_Length, 2)` containing the raw I/Q symbols.
2.  **Y (Targets):** A 2D array containing the variable parameters (`Spacing`, `OSNR`) to be predicted.
3.  **Metadata:** Auxiliary scenario identifiers (`Distance`, `Power`).

In [11]:
# 1. Ensure data is contiguously sorted to align X and Y
sort_cols = ["Distance", "Power", "Spacing", "OSNR", "batch_number"]
df_sorted = df_batches.sort_values(by=sort_cols)

# --- A. Generate X (Raw Symbols) ---
# Reshape flattened data into a 3D structure: (Batches, Sequence_Length, 2)
# This raw data will be used for further feature extraction.
X_symbols = df_sorted[["I", "Q"]].to_numpy(dtype=np.float32).reshape(-1, BATCH_SIZE, 2)

# --- B. Generate Y (Targets) ---
# Since targets are constant per batch, we aggregate to get one row per batch
unique_batches = df_sorted.groupby(sort_cols, sort=False).first().reset_index()

# Extract target variables
Y_targets = unique_batches[["Spacing", "OSNR"]].to_numpy(dtype=np.float32)

# --- C. Generate Metadata (Auxiliary) ---
# Extract scenario identifiers for analysis
M = unique_batches[["Distance", "Power"]].to_numpy(dtype=np.float32)

print(f"X Shape (Raw Symbols): {X_symbols.shape}")
print(f"Y Shape (Targets):     {Y_targets.shape}")
print(f"M Shape (Info):        {M.shape}")

# Quick verification
print("\nFirst batch example:")
print(f"Target (Spacing, OSNR): {Y_targets[0]}")
print(f"Scenario (Dist, Power): {M[0]}")


X Shape (Raw Symbols): (5006, 10000, 2)
Y Shape (Targets):     (5006, 2)
M Shape (Info):        (5006, 2)

First batch example:
Target (Spacing, OSNR): [28.5  26.07]
Scenario (Dist, Power): [0. 0.]


### Signal Analysis: Probabilistic Topology Extraction

In this stage, we transition from raw time-series data to statistical feature vectors. Since the optical signal is subjected to stochastic impairments (ASE noise, linear interchannel interference), we treat the received I/Q symbols as random variables drawn from an unknown probability distribution.

Our goal is to **infer** the underlying structure of this distribution to retrieve the modulation constellation's geometry.

#### 1. Kernel Density Estimation (KDE)
To estimate the Probability Density Function (PDF) $\hat{f}(x)$ of the symbols without assuming a specific parametric family (like a rigid Gaussian Mixture Model), we employ **Kernel Density Estimation**. Given a batch of observed symbols $x_1, x_2, \dots, x_n$, the estimator is defined as:

$$\hat{f}(x) = \frac{1}{nh} \sum_{i=1}^{n} K\left(\frac{x - x_i}{h}\right)$$

Where:
* $K(\cdot)$ is the **Gaussian Kernel**: $K(u) = \frac{1}{\sqrt{2\pi}} e^{-\frac{1}{2}u^2}$.
* $h$ is the **Bandwidth** (smoothing parameter). It controls the bias-variance tradeoff. We include $h$ as a feature because it correlates with the signal's variance (noise power).

#### 2. Topological Inference (Peaks and Valleys)
Once $\hat{f}(x)$ is obtained, we perform a topological analysis to find its critical points.
* **Peaks (Modes):** Correspond to the local maxima of the PDF, satisfying $\hat{f}'(x) = 0$ and $\hat{f}''(x) < 0$. In the context of 16-QAM, these infer the **location of the constellation symbols**. Ideally, for normalized data, these should be at $\{-3, -1, 1, 3\}$.
* **Valleys (Antimodes):** Correspond to the local minima between peaks. These infer the **optimal decision thresholds** for symbol detection.

#### 3. Feature Engineering Logic & Validation
The extraction process follows this pipeline:

1.  **Estimation:** Compute $\hat{f}(x)$ for both In-phase (I) and Quadrature (Q) components.
2.  **Peak/Valley Detection:** Solve for critical points to extract the "skeleton" of the distribution.
3.  **Validation & Logging:** We expect a 4-PAM topology (4 peaks, 3 valleys) per component (I and Q).
    * If `num_peaks < 4`, it implies **modal collapse** due to high noise or distortion (merging of distributions).
    * A **warning is issued** for these batches, flagging them as highly degraded scenarios.
4.  **Sorting & Mapping:** Features are sorted by location ($x$-axis) to preserve the ordinality of modulation levels (e.g., most negative voltage level corresponds to Feature 1).
5.  **Validation:** We verified that all batches successfully resolved to the expected topology (4 peaks). Therefore, no imputation was required, and the redundant count statistics were discarded.

**Final Feature Vector Structure (Per Component):**
The resulting vector $\mathbf{v} \in \mathbb{R}^{10}$ summarizes the statistical shape of the component:
* `bandwidth` of the complete PDF.
* `peaks` (4 locations: $\hat{\mu}_1, \dots, \hat{\mu}_4$), `valleys` (3 locations).

**Total dimensionality:** $2 \times 8 = 16$ statistical features per batch.

#### 3.1 Adaptive Bandwidth Refinement (Iterative KDE)
A common challenge in non-parametric estimation of multimodal distributions is **oversmoothing**. Standard bandwidth selection rules (like Scott's Rule) assume the underlying data is unimodal and Gaussian, often resulting in a bandwidth $h$ that is too large for 4-PAM signals, causing distinct modulation peaks to merge.

To mitigate this bias, we implement an **Adaptive Refinement Strategy**:
1.  Initial estimation is performed using Scott's Rule.
2.  If the number of detected modes ($\hat{M}$) is less than the expected modulation order ($M=4$), we iteratively reduce the bandwidth factor:
    $$h_{new} = h_{old} \times \alpha \quad (\text{where } 0 < \alpha < 1)$$
3.  We re-evaluate $\hat{f}(x)$ and re-count peaks. This loop continues until $\hat{M}=4$ or a minimum bandwidth limit is reached (preventing overfitting to noise).

This approach prioritizes extracting the latent structural topology even in scenarios with moderate signal dispersion.

### 3.2 Visual Diagnosis of Topological Collapse

To empirically validate the robustness of the feature extraction, we implement a visual diagnostic mechanism. For the first $N$ batches where the adaptive algorithm fails to recover the expected 4-PAM topology (i.e., `num_peaks < 4`), we generate a diagnostic plot.

**The plot superimposes:**
1.  **Empirical Histogram:** The discrete frequency distribution of the raw symbols (Ground Truth).
2.  **Refined KDE Curve:** The continuous probability density estimate $\hat{f}(x)$ after bandwidth adaptation.
3.  **Detected Critical Points:**
    * <span style="color:red">**X**</span> Markers: Detected Peaks (Modes).
    * <span style="color:blue">**O**</span> Markers: Detected Valleys (Antimodes).

This visualization allows us to differentiate between **statistical oversmoothing** (where structure exists but is lost by the estimator) and **physical signal degradation** (where the constellation has truly collapsed due to noise).

In [12]:
def plot_kde_diagnosis(data, grid, pdf, peaks_idx, valleys_idx, batch_idx, comp_name, bw_used):
    """
    Generates a diagnostic plot showing Histogram vs KDE vs Peaks.
    """
    plt.figure(figsize=(10, 4))

    # 1. Histogram (Raw Data)
    plt.hist(data, bins=50, density=True, alpha=0.3, color='gray', label='Raw Histogram')

    # 2. KDE Curve
    plt.plot(grid, pdf, color='black', linewidth=2, label=f'Refined KDE (bw={bw_used:.2f})')

    # 3. Peaks and Valleys
    peak_locs = grid[peaks_idx]
    peak_heights = pdf[peaks_idx]
    plt.plot(peak_locs, peak_heights, "x", color='red', markersize=10, markeredgewidth=3, label=f'Peaks ({len(peaks_idx)})')

    valley_locs = grid[valleys_idx]
    valley_heights = pdf[valleys_idx]
    plt.plot(valley_locs, valley_heights, "o", color='blue', markersize=8, fillstyle='none', label=f'Valleys ({len(valleys_idx)})')

    plt.title(f"Diagnostic: Batch {batch_idx} Component {comp_name} | Topology Collapse")
    plt.xlabel("Symbol Amplitude")
    plt.ylabel("Density")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()

def get_adaptive_features(data, batch_idx, comp_name, n_peaks_target=4, n_valleys_target=3, plot_failure=False):
    """
    Extracts KDE features using adaptive bandwidth.
    If plot_failure=True and topology collapses, it plots the diagnosis.
    """
    # 1. Initial KDE
    try:
        kde = gaussian_kde(data)
    except Exception as e:
        warnings.warn(f"Could not compute KDE features for batch {batch_idx}: {e}")
        return np.zeros(1 + n_peaks_target + n_valleys_target), 0

    # Configuration
    max_retries = 3
    reduction_factor = 0.6
    current_bw = kde.factor

    # --- REFINEMENT LOOP ---
    for attempt in range(max_retries + 1):
        kde.set_bandwidth(bw_method=current_bw)

        # Evaluate PDF
        margin = (np.max(data) - np.min(data)) * 0.2
        grid = np.linspace(np.min(data) - margin, np.max(data) + margin, 1000)
        pdf = kde(grid)

        # Find Peaks
        peaks_idx, _ = find_peaks(pdf, height=0)
        num_peaks_found = len(peaks_idx)

        if num_peaks_found >= n_peaks_target:
            break # Success
        else:
            if attempt < max_retries:
                current_bw = current_bw * reduction_factor
            else:
                pass # Failed after all retries

    # --- DIAGNOSTIC PLOTTING ---
    # We plot ONLY if requested AND if the topology is still broken (< 4 peaks)
    if plot_failure and num_peaks_found < n_peaks_target:
        # We find valleys just for the plot
        valleys_idx, _ = find_peaks(-pdf)
        plot_kde_diagnosis(data, kde, grid, pdf, peaks_idx, valleys_idx, batch_idx, comp_name, current_bw)

    # --- FEATURE EXTRACTION (Same as before) ---
    peak_locs = grid[peaks_idx]
    peak_heights = pdf[peaks_idx]

    if num_peaks_found > n_peaks_target:
        top_indices = np.argsort(peak_heights)[::-1][:n_peaks_target]
        selected_locs = peak_locs[top_indices]
    else:
        selected_locs = peak_locs

    final_peaks_vals = np.sort(selected_locs)

    # Valleys
    valleys_idx, _ = find_peaks(-pdf)
    valley_locs = grid[valleys_idx]
    valley_heights = pdf[valleys_idx]
    num_valleys_found = len(valley_locs)

    if num_valleys_found > n_valleys_target:
        deepest_indices = np.argsort(valley_heights)[:n_valleys_target]
        selected_valleys = valley_locs[deepest_indices]
    else:
        selected_valleys = valley_locs

    final_valleys_vals = np.sort(selected_valleys)

    # Padding
    padded_peaks = np.pad(final_peaks_vals, (0, max(0, n_peaks_target - len(final_peaks_vals))), 'constant', constant_values=0.0)
    padded_valleys = np.pad(final_valleys_vals, (0, max(0, n_valleys_target - len(final_valleys_vals))), 'constant', constant_values=0.0)

    # Return features and metadata (num_peaks) for control
    # Features: [bandwidth, peaks (4), valleys (3)] -> Size 8
    features = np.concatenate(([current_bw], padded_peaks, padded_valleys))
    return features, num_peaks_found


In [ ]:
N_PEAKS = 4
N_VALLEYS = N_PEAKS - 1
N_FEATS_SINGLE = 1 + N_PEAKS + N_VALLEYS
TOTAL_FEATS = N_FEATS_SINGLE * 2
processed_features = np.zeros((len(X_symbols), TOTAL_FEATS), dtype=np.float32)

# Plotting control
MAX_DEBUG_PLOTS = 5
plots_generated = 0

print(f"Extracting Adaptive Features on {len(X_symbols)} batches...")
for i, iq_batch in enumerate(X_symbols):
    i_data = iq_batch[:, 0]
    q_data = iq_batch[:, 1]

    # Only enable plotting if we're under MAX_DEBUG_PLOTS
    enable_plot = (plots_generated < MAX_DEBUG_PLOTS)

    with warnings.catch_warnings(record=True) as w:
        warnings.simplefilter("always")

        # Process I component
        feats_i, num_peaks_i = get_adaptive_features(i_data, batch_idx=i, comp_name='I', plot_failure=enable_plot)

        # Verify number of peaks
        if num_peaks_i < 4 and enable_plot:
            plots_generated += 1
            print(f"--> Diagnostic plot generated for Batch {i} (I-Component). Total plots: {plots_generated}/{MAX_DEBUG_PLOTS}")
            enable_plot = (plots_generated < MAX_DEBUG_PLOTS)

        # Process Q component
        feats_q, num_peaks_q = get_adaptive_features(q_data, batch_idx=i, comp_name='Q', plot_failure=enable_plot)

        if num_peaks_q < 4 and enable_plot:
            plots_generated += 1
            print(f"--> Diagnostic plot generated for Batch {i} (Q-Component). Total plots: {plots_generated}/{MAX_DEBUG_PLOTS}")

    processed_features[i, :] = np.concatenate([feats_i, feats_q])

    if (i + 1) % 500 == 0:
        print(f"Processed {i + 1}/{len(X_symbols)} batches.")

print("\nAnalysis complete.")


Extracting Adaptive Features on 5006 batches...
Processed 500/5006 batches.
Processed 1000/5006 batches.
Processed 1500/5006 batches.
Processed 2000/5006 batches.
Processed 2500/5006 batches.
Processed 3000/5006 batches.
Processed 3500/5006 batches.
Processed 4000/5006 batches.
Processed 4500/5006 batches.
Processed 5000/5006 batches.

Analysis complete.


### Summary & Data Export

We have successfully transformed the raw symbol dataset into structured tensors and feature sets ready for machine learning.

**Pipeline Overview:**
1.  **Batching:** Grouped the raw dataset into **5,006** discrete batches of **10,000** symbols, ensuring no mixing between scenarios.
2.  **Raw Tensors:** Created 3D inputs (`X_raw`) and targets (`Y_targets`) for deep learning models.
3.  **Feature Extraction:** Processed the raw signals using robust KDE analysis to extract topological features (Peak/Valley locations), capturing signal degradation and missing modulation levels.

**Final Datasets Saved:**
* `X_features.npy`: The extracted topological features (Bandwidth, Counts, Locations). **Shape:** `(5006, 20)`
* `Y_targets.npy`: The regression targets (Spacing, OSNR). **Shape:** `(5006, 2)`
* `M_metadata.npy`: Scenario identifiers (Distance, Power) for analysis. **Shape:** `(5006, 2)`

In [15]:
# Create a directory to keep things organized
OUTPUT_DIR = "processed_data"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Dictionary map: Filename -> Array
data_artifacts = {
    "X_features": processed_features,
    "Y_targets": Y_targets,
    "M_metadata": M,
}

print(f"Saving datasets to directory: '{OUTPUT_DIR}/'...\n")

for name, array in data_artifacts.items():
    file_path = os.path.join(OUTPUT_DIR, f"{name}.npy")
    np.save(file_path, array)

    # Calculate size in MB for info
    size_mb = array.nbytes / (1024 * 1024)
    print(
        f"✓ Saved {name}.npy".ljust(25)
        + f"| Shape: {str(array.shape).ljust(20)} | Size: {size_mb:.2f} MB"
    )

print("\nAll files saved successfully.")

Saving datasets to directory: 'processed_data/'...

✓ Saved X_features.npy   | Shape: (5006, 20)           | Size: 0.38 MB
✓ Saved Y_targets.npy    | Shape: (5006, 2)            | Size: 0.04 MB
✓ Saved M_metadata.npy   | Shape: (5006, 2)            | Size: 0.04 MB

All files saved successfully.
